In [9]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Lasso

np.random.seed(42)

# ==========================================
# 1. PHYSICS SETUP (Dilute Defect Gas)
# ==========================================
nq = 20
epsilon = 0.1 # Probability of a dipole defect (c=1) in any given state
alpha_x_indices = [0, 1] # The local probe
alpha_z_indices = [4, 5] # The hidden membrane

def generate_ensemble_data(num_states, shots_per_state):
    """Generates (y, b) tuples for an ensemble of random high-degree states"""
    Y_all = []
    b_all = []
    true_labels = []
    
    for _ in range(num_states):
        # 1. Determine if this specific state has a defect (c = 1) or vacuum (c = 0)
        c_n = np.random.choice([0, 1], p=[1 - epsilon, epsilon])
        true_labels.append(c_n)
        
        # 2. The physical quantum shots for this state
        Y_bits = np.random.randint(0, 2, size=(shots_per_state, nq))
        
        # (Assuming the coherent hardware XORs out the nq-1 chaos g(y_inv) perfectly)
        # The hardware returns the topological parity + the defect sector
        z_mask_val = np.sum(Y_bits[:, alpha_z_indices], axis=1) % 2
        b_measured = (z_mask_val + c_n) % 2
        
        Y_all.append(Y_bits)
        b_all.append(b_measured)
        
    return np.array(Y_all), np.array(b_all), np.array(true_labels)

# ==========================================
# 2. TASK 1: TRAINING PHASE (Discovering the Physics)
# ==========================================
print("--- TASK 1: TRAINING PHASE ---")
# We use a training ensemble to discover the membrane alpha_Z
# We assume the training set consists of pristine vacuum states (c=0) for calibration
Y_train, b_train, _ = generate_ensemble_data(num_states=1, shots_per_state=100)

# Flatten for ML
Y_train_flat = Y_train[0]
b_train_flat = b_train[0]

# Convert to physical spins for the ML Fourier Learner
Y_train_spins = 1 - 2 * Y_train_flat
b_train_spins = 1 - 2 * b_train_flat

poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
Y_features_train = poly.fit_transform(Y_train_spins)

# The ML model discovers the underlying symmetry rule
ml_model = Lasso(alpha=0.1)
ml_model.fit(Y_features_train, b_train_spins)

# Extract the learned membrane
learned_mask_indices = []
feature_names = poly.get_feature_names_out([f"Z{i}" for i in range(nq)])
for weight, name in zip(ml_model.coef_, feature_names):
    if abs(weight) > 0.1:
        print(f"✅ ML Discovered the hidden topological rule: {name}")
        # Parse the string (e.g., 'Z4 Z5') back to indices [4, 5]
        indices = [int(idx[1:]) for idx in name.split()]
        learned_mask_indices.extend(indices)

# ==========================================
# 3. TASK 2: INFERENCE PHASE (Defect Classification)
# ==========================================
print("\n--- TASK 2: INFERENCE PHASE (Classifying New States) ---")
# Generate 10 new, completely unseen quantum states. 
# We only use 8 quantum shots per state!
num_test_states = 50
shots = 8 
Y_test, b_test, true_labels = generate_ensemble_data(num_states=num_test_states, shots_per_state=shots)

predictions = []
for n in range(num_test_states):
    # Retrieve the (y, b) data for this specific state
    Y_state = Y_test[n]
    b_state = b_test[n]
    
    # Apply the learned physics rule (evaluate the alpha_Z membrane)
    learned_z_val = np.sum(Y_state[:, learned_mask_indices], axis=1) % 2
    
    # The intercept 'c' is just the XOR difference between the measurement 'b' and the rule
    # c_n = b(y) XOR (alpha_Z * y)
    inferred_c_array = (b_state + learned_z_val) % 2
    
    # Take the majority vote to suppress any residual quantum shot noise
    predicted_label = int(np.round(np.mean(inferred_c_array)))
    predictions.append(predicted_label)

# ==========================================
# 4. RESULTS
# ==========================================
print(f"Testing on {num_test_states} highly-entangled states ({shots} shots each):")
for i in range(num_test_states):
    status = "Match ✅" if predictions[i] == true_labels[i] else "Fail ❌"
    defect_str = "Dipole Defect (c=1)" if true_labels[i] == 1 else "Vacuum (c=0)     "
    print(f"State {i:2d} | True: {defect_str} | ML Predicted: c={predictions[i]} | {status}")

--- TASK 1: TRAINING PHASE ---
✅ ML Discovered the hidden topological rule: Z4 Z5

--- TASK 2: INFERENCE PHASE (Classifying New States) ---
Testing on 50 highly-entangled states (8 shots each):
State  0 | True: Dipole Defect (c=1) | ML Predicted: c=1 | Match ✅
State  1 | True: Dipole Defect (c=1) | ML Predicted: c=1 | Match ✅
State  2 | True: Dipole Defect (c=1) | ML Predicted: c=1 | Match ✅
State  3 | True: Vacuum (c=0)      | ML Predicted: c=0 | Match ✅
State  4 | True: Vacuum (c=0)      | ML Predicted: c=0 | Match ✅
State  5 | True: Vacuum (c=0)      | ML Predicted: c=0 | Match ✅
State  6 | True: Vacuum (c=0)      | ML Predicted: c=0 | Match ✅
State  7 | True: Vacuum (c=0)      | ML Predicted: c=0 | Match ✅
State  8 | True: Vacuum (c=0)      | ML Predicted: c=0 | Match ✅
State  9 | True: Vacuum (c=0)      | ML Predicted: c=0 | Match ✅
State 10 | True: Vacuum (c=0)      | ML Predicted: c=0 | Match ✅
State 11 | True: Vacuum (c=0)      | ML Predicted: c=0 | Match ✅
State 12 | True: Dip